In [1]:
# Imports
import sys
from pathlib import Path

# Resolve project root and ensure it's on sys.path
ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "pyproject.toml").exists() or (ROOT / "raw_data").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import discretize_preprocess

In [2]:
# Preprocess data
from pathlib import Path

dataset_path = ROOT / "raw_data" / "adult.csv"
output_path = ROOT / "discretized_data" / "adult.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path), bins=10, strategy='uniform')

Preprocessing: /home/adity/github/katabatic-mentorship-repo/raw_data/adult.csv
Saved preprocessed discrete dataset to: /home/adity/github/katabatic-mentorship-repo/discretized_data/adult.csv


In [3]:
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.fairtabdiffusion_alex.adapter import KatabaticFairTabDiffusion

# Set paths
input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "adult")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "adult" / "fairtabdiffusion")

# select protected and target attributes.
protected_col = input("Protected Attribute (S): ").strip() or "sex"
target_col = input("Target Attribute (Y): ").strip() or "class"

#FairTabDiffusion parameters
model_config = {
    "epochs": 100,
    "batch_size": 256,
    
    # Fairness Config
    "fairness_config": {
        "S": protected_col, #protected attributed
        "Y": target_col,    #target attribute
        "S_under": "0",     #underrepresented class
        "Y_desire": "1"     #desired representation
    }
}

pipeline = TrainTestSplitPipeline(model=KatabaticFairTabDiffusion)

pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
    **model_config
)

/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Loading FairTabDiffusion data from: /home/adity/github/katabatic-mentorship-repo/sample_data/adult
Initialising FairTabDiffusion (Features:14, Epochs:100)...


Training FairTabDiffusion: 100%|██████████| 100/100 [03:44<00:00,  2.25s/it, loss=0.701]


Generating synthetic data to: /home/adity/github/katabatic-mentorship-repo/synthetic/adult/fairtabdiffusion
Saved artifacts: x_synth.csv, y_synth.csv (1000 rows)


/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/adult/fairtabdiffusion_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7063
F1 Score: 0.6498
AUC: 0.4357

MLP:
Accuracy: 0.6805
F1 Score: 0.6375
AUC: 0.5021

RF:
Accuracy: 0.6968
F1 Score: 0.6491
AUC: 0.4479

XGBoost:
Accuracy: 0.6509
F1 Score: 0.6384
AUC: 0.5014


'Train test split pipeline executed successfully.'